In [1]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_large_table
import time

spark=get_spark("Case03_Partitioning")
df = generate_large_table(spark, num_rows=1_000_000)



26/04/12 10:54:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


#  BAD: Write without partitioning

In [2]:
df.write.mode("overwrite").parquet("/tmp/events_no_partition")
start=time.time()
result_bad=(
    spark.read.parquet("/tmp/events_no_partition")
    .filter(F.col("event_date")=="2025-01-15")
    .agg(F.sum("download_bytes"))
)
result_bad.show()
print(f"Full scan: {time.time() - start:.1f}s")

26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/12 11:03:11 WARN MemoryManager: Total allocation exceeds 95.

+-------------------+
|sum(download_bytes)|
+-------------------+
|         5559569942|
+-------------------+

Full scan: 1.1s


# GOOD: Write partitioned by event_date

In [3]:
df.write.mode("overwrite").partitionBy("event_date").parquet("/tmp/events_partitioned")
start=time.time()
result_good=(
    spark.read.parquet("/tmp/events_partitioned")
    .filter(F.col("event_date") == "2025-01-15")
    .agg(F.sum("download_bytes"))
)
result_good.show()
print(f" Partition pruning: {time.time() - start:.1f}s")

26/04/12 11:14:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/12 11:15:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


+-------------------+
|sum(download_bytes)|
+-------------------+
|         5559569942|
+-------------------+

 Partition pruning: 0.8s


# Best for: filtering (WHERE conditions)